# **Tilt Shift Efekti**

- In this lesson we'll go through some code that generates our Titl Shift effect on our sample images

Source - https://github.com/powersj/tilt-shift

In [ ]:
import cv2
import sys
import numpy as np
from matplotlib import pyplot as plt

def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()


Blend - Maskeye göre iki görüntüyü karıştırmak anlamına gelir.

Örneğin: 

Maskedeki beyaz alan → bulanık görüntü daha baskın

Maskedeki siyah alan → orijinal görüntü daha baskın

Matematiksel olarak: out=mask×blurred+(1−mask)×original

Yani “blend etmek = maskeye göre iki görüntüyü oranlı olarak birleştirmek” demektir.

In [ ]:
# Kodda maskeye göre bir görüntüyü bulanıklaştırıp tilt-shift efekti uygulanıyor.

# --- Resimleri oku ---
image = cv2.imread('../files/images/america32.jpg')           # Orijinal resim
mask = cv2.imread('../files/images/city_mask.jpg', cv2.IMREAD_GRAYSCALE)  # Maske (beyaz = bulanık, siyah = keskin)

#image = cv2.imread('../files/images/boat.jpg')           # Orijinal resim
#mask = cv2.imread('../files/images/boat_mask.jpg', cv2.IMREAD_GRAYSCALE)  # Maske (beyaz = bulanık, siyah = keskin)

imshow('Orijinal', image)
# --- Maske ile orijinal görüntü aynı boyutta değilse yeniden boyutlandır ---
if mask is None:
    raise ValueError('Maske dosyası yüklenemedi. Dosya yolunu kontrol edin.')
if image is None:
    raise ValueError('Görüntü dosyası yüklenemedi. Dosya yolunu kontrol edin.')
if mask.shape != image.shape[:2]:
    # cv2.resize expects (width, height) = (image.shape[1], image.shape[0])
    mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)
imshow('Mask', mask)

# --- Görüntüyü float formatına çevir ---
image = image.astype(np.float32)
mask = mask.astype(np.float32) / 255.0    # 0-1 aralığına normalize et

# --- Görüntüyü bulanıklaştır ---
blurred = cv2.GaussianBlur(image, (21, 21), 0)  # 21x21 kernel ile blur

# --- Basit maskeye göre blend et ---
mask_3ch = cv2.merge([mask, mask, mask])        # 3 kanala dönüştür
out = mask_3ch * blurred + (1 - mask_3ch) * image  

# --- Sonucu uint8 formatına çevir ---
out = np.clip(out, 0, 255).astype(np.uint8)

# --- göster ---
cv2.imwrite('america32_tiltshift.jpg', out)

imshow('Tilt Shift', out)
